# California Wildfire Detections: Exploration

Exploring `fire_data_2010.csv`, data from the **MODIS satellite fire detections over California in 2010**.

**Contents**
1. Imports and Setup
2. Understanding the data (what each column means, min/max, counties, satellites, confidence, brightness, type)
3. What counts as an actual wildfire? (filters)

https://www.fire.ca.gov/incidents/ - link to claifornia fire data, has csv saying whether a fire was a wildfire or not. (could be used to verify wildfires but the csv is only from 2013)

## 1. Imports and Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import folium
import branca.colormap as cm

# ---------- Files ----------
CSV_PATH = Path("Copy of all data") / "fire_data_2010.csv"
OUTPUT_DIR = Path("wildfire_2010_outputs")   # plots (PNG) and the HTML map are saved here
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- What counts as a "wildfire"? (see section 3) ----------
VEGETATION_ONLY = True   # keep only type == 0 (presumed vegetation fire)
MIN_CONFIDENCE  = 0      # 0 = keep all. MODIS C6 User Guide, Table 10: >= 80 high, 30-79 nominal, < 30 low
MIN_FRP         = 0      # minimum Fire Radiative Power (MW) for the plots in section 4 (0 = keep all) 

In [2]:
# Load the raw dataas a dataFrame. parse_dates turns acq_date into proper dates for further analysis. 
raw = pd.read_csv(CSV_PATH, parse_dates=["acq_date"])

# The first column ("Unnamed: 0") is just a row index left over from the source file. (in the csv file) 
raw = raw.drop(columns=["Unnamed: 0"])

print(f"{len(raw):,} detections, {raw.shape[1]} columns")
#raw.head()

2,951 detections, 27 columns


## 2. Understanding the data

Each row is **one hot-spot detection**: a ~1 km pixel that the MODIS sensor judged to be unusually hot. One real fire can produce many rows (several pixels and several detections over several days).

### What each column means

| Column | Meaning |
|---|---|
| `latitude`, `longitude` | Centre of the 1 km detection pixel |
| `brightness` | Brightness temperature (Kelvin) of the pixel in MODIS channel 21/22, a mid-infrared band sensitive to fire. Hotter = brighter |
| `bright_t31` | Brightness temperature (Kelvin) in channel 31 (thermal infrared), which is a better measure of the *background* surface temperature |
| `scan`, `track` | Pixel size across / along the satellite's path (km). Pixels are larger at the edge |
| `acq_date`, `acq_time` | Date and time (UTC, `HHMM`) the satellite saw the detection |
| `satellite`, `instrument` | `Aqua` or `Terra` (both carry the MODIS instrument) |
| `confidence` | 0-100 % confidence that the detection is a real fire (see below) |
| `version` | Processing version of the product (6.03 for every row) |
| `frp` | **Fire Radiative Power** in megawatts (MW): the heat energy the fire radiates per second. The main measure of fire intensity |
| `daynight` | `D` = daytime overpass, `N` = night-time overpass |
| `type` | Kind of hot spot (discussed below) |
| `OBJECTID`, `COUNTY_NAME`, `COUNTY_ABBREV`, `COUNTY_NUM`, `COUNTY_CODE`, `COUNTY_FIPS`, `ISLAND`, `GlobalID`, `SHAPE_Length`, `SHAPE_Area` | Information assumed to be from a California county boundary file. It tells which county each detection falls in. `ISLAND` is only filled for one Channel Islands row; `SHAPE_*` describe the county, not the fire |

Difference between brightness' - is used to spot fires. ("FRP is calculated from the excess mid-infrared radiance a fire produces above its background").

For future analysis and training models, many variables (columns) can be dropped - just need to know where and when a fire occured. 

Information in the table is from: https://www.earthdata.nasa.gov/data/tools/firms/active-fire-data-attributes-modis-viirs

In [3]:
# Column types and missing values
# raw.info()
# No missing values 

### Minimum and maximum of the numeric columns

In [4]:
#key_numeric = ["brightness", "bright_t31","confidence", "frp"]
#raw[key_numeric].agg(["min", "median", "mean", "max"]).T.round(2)

Big difference in frp.
It is very skewed, the median is only ~20 MW but the maximum is ~1,954 MW. 
Therefore, a few huge fires dominate the totals. 

Confidence has a mean value of 67.73

Brightness never goes below 300K because MODIS only reports pixels that are already hot enough to look like a possible fire.

### Which county has the most detections?

In [5]:
print("Counties in the data:", raw["COUNTY_NAME"].nunique())
county_counts = raw["COUNTY_NAME"].value_counts()
print(county_counts.head(10).to_string())
print(f"\nMost detections: {county_counts.index[0]} ({county_counts.iloc[0]} rows, "
      f"{county_counts.iloc[0] / len(raw):.1%} of all detections)")

Counties in the data: 56
COUNTY_NAME
Fresno      416
Kern        193
Tulare      160
Imperial    154
Lassen      108
Tuolumne    104
Colusa       98
Siskiyou     88
Sutter       84
Glenn        79

Most detections: Fresno (416 rows, 14.1% of all detections)


**Fresno** has the most detections (416), well ahead of Kern (193) and Tulare (160). These are Central Valley agricultural counties, so many detections may be crop or agricultural burning rather than wildfires. Therefore need to look at filtering the data.

### What does `confidence` mean?

`confidence` (0-100 %) is the algorithm's estimate of how likely a detection is to be a real fire rather than a false alarm. The MODIS Collection 6 Fire User's Guide (Table 10, section 8.4.3) groups it as:

- **< 30 %**: low confidence
- **30-79 %**: nominal confidence
- **>= 80 %**: high confidence

**How is `confidence` calculated?** It is a *heuristic score*, not a probability. For each detected fire pixel the algorithm combines up to five sub-scores (how hot the pixel is at 4 µm, how much it stands out from its surroundings, the 4 µm vs 11 µm temperature difference, and penalties for neighbouring cloud and water) into a single 0-100 % value (Giglio et al. 2016, section 3.7). The paper does not present it as a calibrated probability, therefore it can be treated as a ranking with 90 being more convincing than 60. The low/nominal/high bands above are cut-offs applied to that score (User's Guide, Table 10). However, NASA FIRMS notes there is no single best cut-off.

**References**
- Giglio, L., Schroeder, W., Hall, J. V., & Justice, C. O. (2018). *MODIS Collection 6 Active Fire Product User's Guide, Revision B*. University of Maryland / NOAA, December 2018. [PDF](https://www.earthdata.nasa.gov/sites/default/files/imported/MODIS_C6_Fire_User_Guide_B.pdf). Section 5.1.3 and Table 10 (confidence classes). 
- Giglio, L., Schroeder, W., & Justice, C. O. (2016). The Collection 6 MODIS active fire detection algorithm and fire products. *Remote Sensing of Environment*, 178, 31-41. [PDF](https://www.earthdata.nasa.gov/s3fs-public/imported/Giglio2016_C6_Fire.pdf). Section 3.7 (detection confidence).
- Collection 6.1 (which replaced Collection 6) did not change the confidence calculation or class thresholds: Giglio et al. (2021), *MODIS Collection 6 and Collection 6.1 Active Fire Product User's Guide, Version 1.0*, [PDF](https://earthdata.nasa.gov/s3fs-public/2023-09/MODIS_C6_C6.1_Fire_User_Guide_1.0.pdf).

In [6]:
conf_band = pd.cut(raw["confidence"], bins=[-1, 29, 79, 100],
                   labels=["Low (<30)", "Nominal (30-79)", "High (>=80)"])
conf_summary = conf_band.value_counts().sort_index().to_frame("detections")
conf_summary["share"] = (conf_summary["detections"] / len(raw)).map("{:.1%}".format)
conf_summary

,detections,share
confidence,,
Low (<30),140,4.7%
Nominal (30-79),1882,63.8%
High (>=80),929,31.5%


Most detections (~64 %) are nominal confidence, about 31 % are high confidence and ~5 % are low. `MIN_CONFIDENCE` in the setup cell can be changed to drop the weaker detections.

### What does `type` mean?

| `type` | Meaning |
|---|---|
| 0 | Presumed vegetation fire |
| 1 | Active volcano (none in this file) |
| 2 | Other static land source |
| 3 | Offshore detection (e.g. oil platforms) |

In [7]:
raw["type"].value_counts().rename_axis("type").to_frame("detections")

,detections
type,
0,2776
2,174
3,1


2,776 rows are type 0, 174 are type 2 and just 1 is type 3.

Reference: NASA FIRMS, [Active fire data attributes (MODIS/VIIRS)](https://earthdata.nasa.gov/data/tools/firms/active-fire-data-attributes-modis-viirs).

In [8]:
offshore = raw[raw["type"] == 3]
offshore[["latitude", "longitude", "acq_date", "acq_time", "frp", "confidence"]]

,latitude,longitude,acq_date,acq_time,frp,confidence
2722,33.9641,-119.7191,2010-05-06,1839,14.5,40


 (offshore, near the Channel Islands in Santa Barbara county). Can filter just vegentation fires using `VEGETATION_ONLY` in the setup cell.


## 3. What counts as an actual wildfire?

There is **no official FRP threshold** that turns a satellite hot spot into a "wildfire". MODIS only detects hot pixels; some are not wildfires at all. Filter ideas:

1. **`type == 0`**: removes industrial, offshore and other non-vegetation heat sources.
2. **`confidence >= 80`** (or at least 30): removes likely false alarms.
3. **FRP above a threshold**: keeps only big/intense fires, in this dataset:
    - median FRP = ~20 MW
    - 90th percentile = ~114 MW (only 336 rows, ~11 %, are >= 100 MW)

Some sources define low-intensity fires as FRP < 50 MW, therefore it may be okay to assume that small agricultural or prescribed burns could sit below ~50 MW?
https://www.researchgate.net/publication/325483817_Deforestation-Induced_Fragmentation_Increases_Forest_Fire_Occurrence_in_Central_Brazilian_Amazonia

To see if agricultural burning could also look at season/place, land cover and how long the fire lasts. This would help tell if it was a wildfire. 
Crop burning happens after harvest before planting

Could use clustering (DBSCAN?) as a wildfire is normally many pixels burning over several days, more persistent than crop burning. 

The interactive map in the plots notebook also shows that some small detections are in the corner of fields - implies crop burning. 